In [ ]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

# BPE FOR GPT-2 MODEL

In [ ]:
print(tokenizer.n_vocab) 

In [ ]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

In [ ]:
strings = tokenizer.decode(integers)

print(strings)

**Let us take another simple example to illustrate how the BPE tokenizer deals with unknown tokens**

In [ ]:
integers = tokenizer.encode("Akwirw ier")
print(integers)

strings = tokenizer.decode(integers)
print(strings)

 ### CREATING INPUT-TARGET PAIRS

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))  # vocab size for this verdict dataset after applying BPE
# Executing the code above will return 5145, the total number of tokens in the training set, after applying the BPE tokenizer.

In [ ]:
enc_sample = enc_text[50:]
# Next, we remove the first 50 tokens from the dataset for demonstration purposes as it
# results in a slightly more interesting text passage in the next steps

**One of the easiest and most intuitive ways to create the input-target pairs for the nextword prediction task is to create two variables, x and y, where x contains the input tokens
and y contains the targets, which are the inputs shifted by 1**

In [ ]:
## The context size determines how many tokens are included in the input

In [ ]:
context_size = 4 #length of the input
#The context_size of 4 means that the model is trained to look at a sequence of 4 words (or tokens) 
#to predict the next word in the sequence. 
#The input x is the first 4 tokens [1, 2, 3, 4], and the target y is the next 4 tokens [2, 3, 4, 5]

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

In [ ]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "---->", desired)

In [ ]:
# For illustration purposes, let's repeat the previous code but convert the token IDs into text
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

**Implementing an efficient data loader that
iterates over the input dataset and returns the inputs and targets as PyTorch tensors, which
can be thought of as multidimensional arrays.**

### IMPLEMENTING A DATA LOADER

In [ ]:
# Understanding DataSet and DataLoader in PyTorch

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

# 1. THE DATASET (The Vending Machine)
class FruitDataset(Dataset):
    def __init__(self):
        self.fruits = ["Apple", "Banana", "Cherry", "Date", "Elderberry"]
        self.colors = ["Red", "Yellow", "Red", "Brown", "Purple"]

    def __len__(self):
        return len(self.fruits)

    def __getitem__(self, idx):
        # Fetch one item for the index given by the Sampler
        return self.fruits[idx], self.colors[idx]

# 2. THE DATALOADER (The Delivery Person)
my_dataset = FruitDataset()
# Setting batch_size=2 and shuffle=True
loader = DataLoader(my_dataset, batch_size=2, shuffle=True)

# 3. THE USE CASE (The Training Loop)
print("--- Starting Delivery ---")
for i, (fruits, colors) in enumerate(loader):
    print(f"Batch {i+1} delivered: {fruits} (Colors: {colors})")

**For the efficient data loader implementation, we will use PyTorch's built-in Dataset and DataLoader classes**

In [ ]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

**The following code will use the GPTDatasetV1 to load the inputs in batches via a PyTorch**

In [ ]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

**Testing the dataloader with a batch size of 1 for an LLM with a context size of 4**

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

**To illustrate the meaning of stride=1**

In [ ]:
second_batch = next(data_iter)
print(second_batch)

**Batch sizes of 1, such as we have sampled from the data loader so far, are useful for illustration purposes.**

**If you have previous experience with deep learning, you may know that small batch sizes require less memory during training but lead to more noisy model updates.**

**Just like in regular deep learning, the batch size is a trade-off and hyperparameter to experiment with when training LLMs.**

In [ ]:
# data loader to sample with a batch size greater than 1:

In [ ]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

In [ ]:
num_batches = len(dataloader)
print(f"Total batches: {num_batches}")

In [ ]:
#printing all input output paits of batch size 8 and  context size 4 totally they are 160 batches as such
for i, (input_pair, target_pair) in enumerate(dataloader):
    print(f"\nBatch {i+1}")
    for inp, tgt in zip(input_pair, target_pair):
        print(f"{str(inp):<30} -> {tgt}")

In [ ]:
for i, (input_pair, target_pair) in enumerate(dataloader):
    print(f"Batch {i+1} {input_pair},{target_pair}")

**Note that we increase the stride to 4. This is to utilize the data set fully (we don't skip a
single word) but also avoid any overlap between the batches, since more overlap could lead
to increased overfitting.**

# Vector Embeddings

In [ ]:
# firstly understanding about basics of vector embeddings
# Import trained model
# !pip install gensim


In [ ]:
# import gensim.downloader as api
# model = api.load("word2vec-google-news-300")  # download the model and return as object ready for use

In [ ]:
# # Example of a word as a vector
# word_vectors=model

# # Let us look how the vector embedding of a word looks like
# print(word_vectors['computer'])  # Example: Accessing the vector for the word 'computer'

In [ ]:
# # Similar words
# # Example of using most_similar
# print(word_vectors.most_similar(positive=['king', 'woman'], negative=['man'], topn=10))


In [ ]:
# # Let us check the similarity b/w a few pair of words
# # Example of calculating similarity
# print(word_vectors.similarity('woman', 'man'))
# print(word_vectors.similarity('king', 'queen'))
# print(word_vectors.similarity('uncle', 'aunt'))
# print(word_vectors.similarity('boy', 'girl'))
# print(word_vectors.similarity('nephew', 'niece'))
# print(word_vectors.similarity('paper', 'water'))

In [ ]:
# import numpy as np
# # Words to compare
# word1 = 'man'
# word2 = 'woman'

# word3 = 'semiconductor'
# word4 = 'earthworm'

# word5 = 'nephew'
# word6 = 'niece'

# # Calculate the vector difference
# vector_difference1 = model[word1] - model[word2]
# vector_difference2 = model[word3] - model[word4]
# vector_difference3 = model[word5] - model[word6]

# # Calculate the magnitude of the vector difference
# magnitude_of_difference1 = np.linalg.norm(vector_difference1)
# magnitude_of_difference2 = np.linalg.norm(vector_difference2)
# magnitude_of_difference3 = np.linalg.norm(vector_difference3)


# # Print the magnitude of the difference
# print("The magnitude of the difference between '{}' and '{}' is {:.2f}".format(word1, word2, magnitude_of_difference1))
# print("The magnitude of the difference between '{}' and '{}' is {:.2f}".format(word3, word4, magnitude_of_difference2))
# print("The magnitude of the difference between '{}' and '{}' is {:.2f}".format(word5, word6, magnitude_of_difference3))

### CREATING TOKEN EMBEDDINGS

In [ ]:
input_ids = torch.tensor([2, 3, 5, 1])

In [ ]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [ ]:
print(embedding_layer.weight)

**We can see that the weight matrix of the embedding layer contains small, random values. These values are optimized during LLM training as part of the LLM optimization itself,Moreover, we can see that the weight matrix has six rows and three columns. There is one row for each of the six possible tokens in the vocabulary. And there is one column for each of the three embedding dimensions.**

In [ ]:
print(embedding_layer(torch.tensor([3])))

In [ ]:
print(embedding_layer(input_ids))
